## 04_bias_model — 線形 bias モデル検証

**入力**
- `output/xi/xi_cc.csv`       — ξ_CC(r)
- `output/xi/xi_ww.csv`       — ξ_WW(r)
- `output/xi/xi_cw_wide.csv`  — ξ_CW(r)

**出力**
- `output/analysis/bias_model.csv` — r_km, xi_cc, xi_ww, xi_cw, xi_cw_pred, residual, bw_bc

**前提**
- `01_xi_cc.ipynb`, `01_xi_ww.ipynb`, `02_xi_cw.ipynb` 実行済み

**モデル**
$$\xi_{CW}^{\rm pred}(r) = \sqrt{\xi_{CC}(r) \cdot \xi_{WW}(r)}, \quad
\frac{b_W}{b_C}(r) = \sqrt{\frac{\xi_{WW}(r)}{\xi_{CC}(r)}}$$

残差 $\Delta\xi_{CW} = \xi_{CW} - \xi_{CW}^{\rm pred}$ が0に近いほど線形 bias モデルが成立。

In [1]:
%use dataframe
%use lets-plot

import kotlin.math.*

In [2]:
// --- CSV 読み込み ---
val dfCC = DataFrame.readCSV("./output/xi/xi_cc.csv")
val dfWW = DataFrame.readCSV("./output/xi/xi_ww.csv")
val dfCW = DataFrame.readCSV("./output/xi/xi_cw_wide.csv")

data class BiasBin(
    val rCenter: Double,
    val xiCC: Double, val xiWW: Double, val xiCW: Double,
    val xiPred: Double,   // sqrt(xiCC * xiWW)
    val residual: Double, // xiCW - xiPred
    val bwBc: Double      // b_W/b_C = sqrt(xiWW/xiCC)
)

// ビン数を CC で揃える（WW は meanNN が異なりビン数が変わる可能性あり）
val nBins = dfCC.rowsCount()
println("nBins (CC) = $nBins  |  nBins (WW) = ${dfWW.rowsCount()}  |  nBins (CW) = ${dfCW.rowsCount()}")

nBins (CC) = 38  |  nBins (WW) = 39  |  nBins (CW) = 38


In [3]:
// --- bias モデル計算 ---
// WW と CW は CC と r_km が異なる可能性があるため、CC の r に最近傍 WW/CW を紐付ける
val ccRows = dfCC.rows().toList()
val wwRows = dfWW.rows().toList()
val cwRows = dfCW.rows().toList()

// 最近傍マッチング（r_km の差が最小のビン）
fun nearestRow(rows: List<org.jetbrains.kotlinx.dataframe.DataRow<*>>, r: Double) =
    rows.minByOrNull { abs((it["r_km"] as Double) - r) }!!

val biasResult: List<BiasBin> = ccRows.map { ccRow ->
    val r  = ccRow["r_km"] as Double
    val cc = ccRow["xi"]   as Double
    val ww = (nearestRow(wwRows, r)["xi"] as Double)
    val cw = (nearestRow(cwRows, r)["xi"] as Double)
    val pred   = if (cc > 0 && ww > 0) sqrt(cc * ww) else Double.NaN
    val resid  = if (pred.isNaN()) Double.NaN else cw - pred
    val bwbc   = if (cc > 0 && ww > 0) sqrt(ww / cc) else Double.NaN
    BiasBin(r, cc, ww, cw, pred, resid, bwbc)
}

println("%-8s  %-8s  %-8s  %-8s  %-8s  %-10s  %-8s"
    .format("r [km]", "ξ_CC", "ξ_WW", "ξ_CW", "pred", "Δξ_CW", "b_W/b_C"))
println("-".repeat(70))
biasResult.forEach { b ->
    println("%-8.1f  %-8.4f  %-8.4f  %-8.4f  %-8.4f  %-10.5f  %-8.4f"
        .format(b.rCenter, b.xiCC, b.xiWW, b.xiCW,
                if (b.xiPred.isNaN()) Double.NaN else b.xiPred,
                if (b.residual.isNaN()) Double.NaN else b.residual,
                if (b.bwBc.isNaN()) Double.NaN else b.bwBc))
}

r [km]    ξ_CC      ξ_WW      ξ_CW      pred      Δξ_CW       b_W/b_C 
----------------------------------------------------------------------
8.2       232.3009  212.2762  215.1547  222.0629  -6.90819    0.9559  
9.5       251.8541  217.6798  216.2987  234.1443  -17.84568   0.9297  
11.0      217.6717  187.8567  203.0214  202.2155  0.80593     0.9290  
12.7      187.0023  166.9037  174.6232  176.6674  -2.04417    0.9447  
14.8      181.6368  158.5104  164.7304  169.6800  -4.94962    0.9342  
17.1      164.9913  148.1831  156.8328  156.3615  0.47130     0.9477  
19.8      142.4958  132.9559  132.9805  137.6432  -4.66275    0.9659  
22.9      120.7910  108.5271  111.0481  114.4950  -3.44686    0.9479  
26.5      100.0610  89.6974   91.8044   94.7376   -2.93321    0.9468  
30.7      76.6338   71.2544   70.8931   73.8952   -3.00204    0.9643  
35.5      55.8458   52.3632   52.8462   54.0765   -1.23027    0.9683  
41.1      42.4625   37.0610   39.0231   39.6699   -0.64685    0.9342  
47.6  

In [4]:
// --- プロット1: ξ_CW 実測 vs 幾何平均予測 ---
val valid = biasResult.filter { !it.xiPred.isNaN() && it.xiCC > 0 && it.xiWW > 0 }
val rV   = valid.map { it.rCenter }
val cwV  = valid.map { it.xiCW }
val predV = valid.map { it.xiPred }
val rLong  = rV + rV
val xiLong = cwV + predV
val lbl    = List(rV.size) { "ξ_CW 実測" } + List(rV.size) { "√(ξ_CC·ξ_WW) 予測" }

letsPlot(mapOf("r" to rLong, "xi" to xiLong, "lbl" to lbl)) +
    geomLine(size = 1.2) { x = "r"; y = "xi"; color = "lbl" } +
    geomPoint(size = 1.8) { x = "r"; y = "xi"; color = "lbl" } +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "r [km]") + scaleYContinuous(name = "ξ(r)") +
    scaleColorManual(values = mapOf("ξ_CW 実測" to "#E87040", "√(ξ_CC·ξ_WW) 予測" to "#4682B4")) +
    ggtitle("線形 bias モデル検証: ξ_CW vs √(ξ_CC·ξ_WW)") +
    ggsize(800, 400)

<path d="M0.0 38.27900497069612 L0.0 38.27900497069612 L15.629988555606388 36.852055798796016 L31.259977111212777 53.414426260290526 L46.88996566681911 88.83900039269005 L62.519954222425525 101.17955945540515 L78.14994277803189 111.03122525369332 L93.77993133363825 140.785251354666 L109.4099198892446 168.14421552147815 L125.03990844485097 192.1493158478873 L140.66989700045738 218.23450921796956 L156.29988555606374 240.74672105896525 L171.92987411167016 257.9900235560941 L187.55986266727652 269.89850532894536 L203.18985122288288 279.51713076316025 L218.81983977848924 286.74789259941974 L234.4498283340956 292.1226591498787 L250.07981688970196 296.4469706982474 L265.70980544530835 299.61134662166 L281.3397940009147 300.1136808847407 L296.96978255652107 301.7466803201776 L312.5997711121274 303.0828647168115 L328.2297596677339 304.4707152742201 L343.85974822334015 304.9458459155582 L359.4897367789466 304.3077212103083 L375.119725334553 305.62915246893306 L390.74971389015934 305.92347299882834 L406.3797024457657 305.87024978119734 L422.00969100137195 305.72742559628665 L437.6396795569784 306.09758059882984 L453.2696681125849 305.65867572003447 L468.89965666819114 302.8227062702429 L484.5296452237975 304.82007036917526 L500.159633779404 306.299219559944 L531.4196108906167 306.40909090909093 L547.049599446223 306.0129444965851 " fill="none" stroke-width="2.64" stroke="rgb(232,112,64)" stroke-opacity="1.0">
 
 
 
 <path d="M0.0 29.661558761767424 L0.0 29.661558761767424 L15.629988555606388 14.590909090909065 L31.259977111212777 54.419759656516135 L46.88996566681911 86.28904671877368 L62.519954222425525 95.00527471292139 L78.14994277803189 111.61913938240818 L93.77993133363825 134.96882133946866 L109.4099198892446 163.84451961165936 L125.03990844485097 188.4903581519871 L140.66989700045738 214.48969252842082 L156.29988555606374 239.21205321665332 L171.92987411167016 257.1831257686011 L187.55986266727652 269.03272411985824 L203.18985122288288 279.0551583603533 L218.81983977848924 286.43796672381137 L234.4498283340956 292.3355051878824 L250.07981688970196 296.3824242406021 L265.70980544530835 299.7391435604368 L281.3397940009147 300.0510495761849 L296.96978255652107 301.7626633714786 L312.5997711121274 303.0353355102946 L328.2297596677339 304.4473515870977 L343.85974822334015 304.9870290112598 L359.4897367789466 304.25973815179555 L375.119725334553 305.60230981230575 L390.74971389015934 305.95259075224027 L406.3797024457657 305.88460683637254 L422.00969100137195 305.7574738316068 L437.6396795569784 306.071305705436 L453.2696681125849 305.69357919411675 L468.89965666819114 302.9079771921831 L484.5296452237975 304.74201798779984 L500.159633779404 306.3019553239961 L531.4196108906167 306.4034151888841 L547.049599446223 306.0130358709488 " fill="none" stroke-width="2.64" stroke="rgb(70,130,180)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 線形 bias モデル検証: ξ_CW vs √(ξ_CC·ξ_WW) 
 
 
 
 
 ξ(r) 
 
 
 
 
 r [km] 
 
 
 
 
 
 
 
 
 lbl 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ξ_CW 実測 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 √(ξ_CC·ξ_WW) 予測

In [5]:
// --- プロット2: b_W/b_C(r) + Δξ_CW(r) ---
val bwBcV  = valid.map { it.bwBc }
val residV = valid.map { it.residual }

val p1 = letsPlot(mapOf("r" to rV, "bwbc" to bwBcV)) +
    geomLine(color = "#8B1A1A", size = 1.2) { x = "r"; y = "bwbc" } +
    geomHLine(yintercept = 1.0, linetype = "dashed", color = "#888888") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "r [km]") + scaleYContinuous(name = "b_W / b_C") +
    ggtitle("スケール依存 bias 比  b_W/b_C(r)",
            "> 1 → Woolworths 高バイアス | < 1 → Coles 高バイアス") +
    ggsize(800, 300)

p1

<path d="M0.0 168.09573268178872 L0.0 168.09573268178872 L20.858894269631378 179.9212418347995 L41.71778853926281 180.23188570097767 L62.57668280889419 173.1394095457478 L83.43557707852563 177.89816248290657 L104.29447134815706 171.8049621986146 L125.15336561778844 163.5819284943575 L146.01225988741987 171.72358358378597 L166.87115415705125 172.20922544074944 L187.73004842668269 164.34017987585355 L208.58894269631412 162.51336478808764 L229.4478369659455 177.87061637183535 L250.30673123557688 192.81818181818193 L271.16562550520837 191.5730997102463 L292.02451977483975 169.8187745401696 L312.8834140444711 132.92129653433756 L333.7423083141025 120.65689830683272 L354.601202583734 125.18864246576521 L375.46009685336537 146.7435289985217 L396.31899112299675 111.09062118394456 L417.17788539262824 126.36952178391209 L438.0367796622596 92.27968086660513 L458.8956739318909 95.6789920581524 L479.7545682015225 130.8770599977334 L500.61346247115375 58.58044762607324 L521.4723567407852 9.181818181818358 L542.3312510104167 99.96504266904071 L563.190145280048 80.07832582495541 L584.0490395496794 64.99421302492351 L604.907933819311 87.14391561968193 L625.7668280889422 177.18418149509694 L646.6257223585736 128.1035947776441 L667.4846166282051 39.76882710520567 L709.2024051674679 131.47692620139742 L730.0612994370994 136.59724813928472 " fill="none" stroke-width="2.64" stroke="rgb(139,26,26)" stroke-opacity="1.0">
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 1,000 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 1.1 
 
 
 
 
 
 
 1.2 
 
 
 
 
 
 
 1.3 
 
 
 
 
 
 
 
 
 スケール依存 bias 比 b_W/b_C(r) 
 
 
 
 
 > 1 → Woolworths 高バイアス | < 1 → Coles 高バイアス 
 
 
 
 
 b_W / b_C 
 
 
 
 
 r [km]

In [6]:
// --- CSV 出力 ---
java.io.File("./output/analysis/bias_model.csv").bufferedWriter().use { w ->
    w.appendLine("r_km,xi_cc,xi_ww,xi_cw,xi_cw_pred,residual,bw_bc")
    biasResult.forEach { b ->
        w.appendLine("${b.rCenter},${b.xiCC},${b.xiWW},${b.xiCW},${b.xiPred},${b.residual},${b.bwBc}")
    }
}
println("保存: output/analysis/bias_model.csv")

// BAO スケールのサマリー
val bao = biasResult.minByOrNull { abs(it.rCenter - 666.0) }!!
println()
println("=== Retail BAO (r ≈ ${bao.rCenter.toInt()} km) での bias モデル検証 ===")
println("  ξ_CC         = %.4f".format(bao.xiCC))
println("  ξ_WW         = %.4f".format(bao.xiWW))
println("  ξ_CW 実測    = %.4f".format(bao.xiCW))
println("  ξ_CW 予測    = %.4f".format(bao.xiPred))
println("  残差 Δξ_CW  = %.4f".format(bao.residual))
println("  b_W/b_C      = %.4f".format(bao.bwBc))

保存: output/analysis/bias_model.csv

=== Retail BAO (r ≈ 666 km) での bias モデル検証 ===
  ξ_CC         = 3.2215
  ξ_WW         = 2.8209
  ξ_CW 実測    = 3.0829
  ξ_CW 予測    = 3.0145
  残差 Δξ_CW  = 0.0684
  b_W/b_C      = 0.9358
